In [22]:
from backtesting import Backtest, Strategy
import yfinance as yf
import pandas as pd

In [23]:
df_msft = yf.download('MSFT', period='1y', interval='4h')
df_msft.columns = df_msft.columns.get_level_values(0)
df_msft.dropna(inplace=True)

[*********************100%***********************]  1 of 1 completed


In [24]:
class MSFT_Strategy(Strategy):
    
    sl_pct = 0.015      # 1.5% stop loss
    tp_pct = 0.025      # 2.5% take profit
    vol_window = 20
    vol_threshold = 1.5
    lookback = 20

    def init(self):
        close = pd.Series(self.data.Close)
        returns = close.pct_change() * 100
        
        self.rolling_vol = self.I(
            lambda x: pd.Series(x).rolling(self.vol_window).std(),
            returns
        )
        
        # Percentil 10 (muy bajo)
        self.rolling_p10 = self.I(
            lambda x: pd.Series(x).rolling(self.lookback).quantile(0.10),
            close
        )

    def next(self):
        price = self.data.Close[-1]
        
        if self.position:
            return
        
        vol = self.rolling_vol[-1]
        avg_vol = self.rolling_vol[-self.vol_window:].mean()
        p10 = self.rolling_p10[-1]
        
        # Entra SOLO si:
        # 1. Volatilidad está ALTA
        # 2. Precio está MUY bajo (percentil 10)
        if vol > avg_vol * self.vol_threshold and price < p10:
            self.buy(
                sl=price * (1 - self.sl_pct),
                tp=price * (1 + self.tp_pct)
            )

In [25]:
bt_msft = Backtest(df_msft, MSFT_Strategy, cash=100000, commission=0.0001)
results_msft = bt_msft.run()
print(results_msft)
print(results_msft['_trades'])

Start                     2025-05-02 13:30...
End                       2026-05-01 17:30...
Duration                    364 days 04:00:00
Exposure Time [%]                     1.60966
Equity Final [$]                 100736.25974
Equity Peak [$]                  103556.51738
Commissions [$]                      81.83624
Return [%]                            0.73626
Buy & Hold Return [%]                -7.99656
Return (Ann.) [%]                      0.7392
Volatility (Ann.) [%]                 3.67345
CAGR [%]                              0.50891
Sharpe Ratio                          0.20123
Sortino Ratio                         0.30133
Calmar Ratio                          0.27143
Alpha [%]                             0.92562
Beta                                  0.02368
Max. Drawdown [%]                     -2.7234
Avg. Drawdown [%]                     -2.7234
Max. Drawdown Duration       91 days 03:00:00
Avg. Drawdown Duration       91 days 03:00:00
# Trades                          